In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
from collections import Counter

with open('../results/olmo2_7b_instruct/pretraining/gpt-5-mini/e2_concepts_ranked_standard.json', 'r') as f:
    data = json.load(f)

data[0].keys()

dict_keys(['id', 'prompt', 'semantic_category', 'response', 'model', 'metadata', 'hb_label', 'ranked_concepts', 'ranking_sanity_flags', 'rank_model', 'rank_prompt_version', 'ranked_at', 'stage1_concepts', 'stage1_sanity_flags', 'stage1_extraction_model', 'stage1_prompt_version', 'stage1_extracted_at'])

### 1. Overall Summary Statistics

In [3]:
print("=" * 70)
print("Section 1. Overall Summary Statistics")
print("=" * 70)

total_records = len(data)
total_concepts = sum(len(r["ranked_concepts"]) for r in data)
concept_counts = [len(r["ranked_concepts"]) for r in data]

print(f"  Total records             : {total_records}")
print(f"  Total ranked concepts     : {total_concepts}")
print(f"  Avg concepts per record   : {total_concepts / total_records:.1f}")
print(f"  Min / Max                 : {min(concept_counts)} / {max(concept_counts)}")
print()

# semantic_category distribution
cat_counts = Counter(r["semantic_category"] for r in data)
print("  [Distribution of semantic_category]")
for cat, cnt in sorted(cat_counts.items()):
    print(f"    {cat:<35}: {cnt} records")
print()

# rank_model / rank_prompt_version
rank_models   = set(r.get("rank_model", "?") for r in data)
rank_versions = set(r.get("rank_prompt_version", "?") for r in data)
print(f"  rank_model          : {rank_models}")
print(f"  rank_prompt_version : {rank_versions}")
print()

# Stage 1 provenance
stage1_models   = set(r.get("stage1_extraction_model", "?") for r in data)
stage1_versions = set(r.get("stage1_prompt_version", "?") for r in data)
print(f"  stage1_extraction_model : {stage1_models}")
print(f"  stage1_prompt_version   : {stage1_versions}")
print()

# Per-record concept count
print("  [Number of ranked concepts per record]")
print(f"  {'id':<6} {'semantic_category':<35} {'# ranked':>8} {'# stage1':>8}")
print(f"  {'-'*6} {'-'*35} {'-'*8} {'-'*8}")
for r in data:
    n_ranked = len(r["ranked_concepts"])
    n_stage1 = len(r["stage1_concepts"])
    match_mark = "" if n_ranked == n_stage1 else " ⚠ mismatch"
    print(f"  {r['id']:<6} {r['semantic_category']:<35} {n_ranked:>8} {n_stage1:>8}{match_mark}")

Section 1. Overall Summary Statistics
  Total records             : 6
  Total ranked concepts     : 161
  Avg concepts per record   : 26.8
  Min / Max                 : 20 / 34

  [Distribution of semantic_category]
    chemical_biological                : 1 records
    cybercrime_intrusion               : 1 records
    misinformation_disinformation      : 4 records

  rank_model          : {'gpt-5-mini'}
  rank_prompt_version : {'v1_ranking'}

  stage1_extraction_model : {'gpt-5-mini'}
  stage1_prompt_version   : {'v2_entity_only_id62_krakatoa'}

  [Number of ranked concepts per record]
  id     semantic_category                   # ranked # stage1
  ------ ----------------------------------- -------- --------
  30     misinformation_disinformation             25       25
  31     misinformation_disinformation             23       23
  38     misinformation_disinformation             31       31
  39     misinformation_disinformation             28       28
  182    cybercrime_intrusi

### 2. Centrality Tier Distribution

In [4]:
print("=" * 70)
print("Section 2. Centrality Tier Distribution")
print("=" * 70)

TIER_ORDER = ["topic_core", "primary", "supporting", "peripheral"]

# Global distribution
all_centrality = [c["centrality"] for r in data for c in r["ranked_concepts"]]
global_counts = Counter(all_centrality)
total = sum(global_counts.values())

print("  [Global tier distribution]")
print(f"  {'tier':<15} {'count':>6} {'%':>7}")
print(f"  {'-'*15} {'-'*6} {'-'*7}")
for tier in TIER_ORDER:
    cnt = global_counts.get(tier, 0)
    print(f"  {tier:<15} {cnt:>6} {cnt/total*100:>6.1f}%")
print()

# Per-record tier breakdown
print("  [Per-record tier breakdown]")
print(f"  {'id':<6} {'topic_core':>10} {'primary':>8} {'supporting':>10} {'peripheral':>10}")
print(f"  {'-'*6} {'-'*10} {'-'*8} {'-'*10} {'-'*10}")
for r in data:
    tc = Counter(c["centrality"] for c in r["ranked_concepts"])
    print(f"  {r['id']:<6} {tc.get('topic_core',0):>10} {tc.get('primary',0):>8} "
          f"{tc.get('supporting',0):>10} {tc.get('peripheral',0):>10}")
print()

# Per-record rank range per tier
print("  [Rank range per tier per record]")
for r in data:
    ranked = r["ranked_concepts"]
    tier_ranks = {}
    for c in ranked:
        tier_ranks.setdefault(c["centrality"], []).append(c["rank"])
    parts = []
    for tier in TIER_ORDER:
        if tier in tier_ranks:
            lo, hi = min(tier_ranks[tier]), max(tier_ranks[tier])
            parts.append(f"{tier}:{lo}-{hi}")
    print(f"  id={r['id']}: {' | '.join(parts)}")

Section 2. Centrality Tier Distribution
  [Global tier distribution]
  tier             count       %
  --------------- ------ -------
  topic_core          17   10.6%
  primary             82   50.9%
  supporting          57   35.4%
  peripheral           5    3.1%

  [Per-record tier breakdown]
  id     topic_core  primary supporting peripheral
  ------ ---------- -------- ---------- ----------
  30              3       12          9          1
  31              3       14          4          2
  38              3       16         10          2
  39              3       13         12          0
  182             3        8          9          0
  196             2       19         13          0

  [Rank range per tier per record]
  id=30: topic_core:1-3 | primary:4-15 | supporting:16-24 | peripheral:25-25
  id=31: topic_core:1-3 | primary:4-17 | supporting:18-21 | peripheral:22-23
  id=38: topic_core:1-3 | primary:4-19 | supporting:20-29 | peripheral:30-31
  id=39: topic_core:1-3 | p

#### Centrality Tier Definitions

| Tier | Meaning | Example (record 30: Syrian Civil War) |
|---|---|---|
| topic_core | Central concept of the response; the response would not hold together without it | `Syrian Civil War`, `Assad regime` |
| primary | Important concept directly addressing the topic | `political reform`, `human rights`, `Arab Spring` |
| supporting | Contextual or secondary concept | `Russia`, `Iran`, `United States` (geopolitical background) |
| peripheral | Incidental mention; main narrative holds without it (typically the most generic terms) | — |

### 3. Sanity Flag Analysis

In [5]:
print("=" * 70)
print("Section 3. Sanity Flag Analysis")
print("=" * 70)

# --- ranking_sanity_flags (Stage 2) ---
print("  [Stage 2: ranking_sanity_flags]")
r_flagged = [r for r in data if r["ranking_sanity_flags"]]
r_clean   = [r for r in data if not r["ranking_sanity_flags"]]
print(f"  Clean records : {len(r_clean)}")
print(f"  Flagged records: {len(r_flagged)}")

all_r_flags = [f for r in data for f in r["ranking_sanity_flags"]]
if all_r_flags:
    print("  [Flag type counts]")
    for flag, cnt in Counter(all_r_flags).items():
        print(f"    {flag:<30}: {cnt} times")
    print()
    print("  [Flagged records details]")
    for r in r_flagged:
        print("  " + "-" * 70)
        print(f"  id={r['id']} | semantic_category={r['semantic_category']}")
        print(f"  ranking_sanity_flags: {r['ranking_sanity_flags']}")

        if "missing_concepts" in r["ranking_sanity_flags"]:
            stage1_texts = {c["text"].strip() for c in r["stage1_concepts"]}
            ranked_texts  = {c["text"].strip() for c in r["ranked_concepts"]}
            missing = stage1_texts - ranked_texts
            print(f"  [missing_concepts] ({len(missing)}):")
            for t in sorted(missing):
                print(f"    - \"{t}\"")

        if "extra_concepts" in r["ranking_sanity_flags"]:
            stage1_texts = {c["text"].strip() for c in r["stage1_concepts"]}
            ranked_texts  = {c["text"].strip() for c in r["ranked_concepts"]}
            extra = ranked_texts - stage1_texts
            print(f"  [extra_concepts] ({len(extra)}):")
            for t in sorted(extra):
                print(f"    - \"{t}\"")

        if "duplicate_texts" in r["ranking_sanity_flags"]:
            texts = [c["text"].strip() for c in r["ranked_concepts"]]
            dups = [t for t, cnt in Counter(texts).items() if cnt > 1]
            print(f"  [duplicate_texts] ({len(dups)}):")
            for t in dups:
                print(f"    - \"{t}\"")

        if "rank_not_bijective" in r["ranking_sanity_flags"]:
            ranks = [c["rank"] for c in r["ranked_concepts"]]
            n = len(r["ranked_concepts"])
            expected = set(range(1, n + 1))
            missing_ranks = sorted(expected - set(ranks))
            dup_ranks     = sorted([k for k, v in Counter(ranks).items() if v > 1])
            print(f"  [rank_not_bijective] missing={missing_ranks} duplicated={dup_ranks}")

        if "tier_order_violation" in r["ranking_sanity_flags"]:
            level = {"topic_core": 3, "primary": 2, "supporting": 1, "peripheral": 0}
            sorted_c = sorted(r["ranked_concepts"], key=lambda c: c["rank"])
            print(f"  [tier_order_violation] offending transitions:")
            for i in range(1, len(sorted_c)):
                prev, curr = sorted_c[i-1], sorted_c[i]
                if level.get(curr["centrality"], -1) > level.get(prev["centrality"], -1):
                    print(f"    rank {prev['rank']} ({prev['centrality']}) → "
                          f"rank {curr['rank']} ({curr['centrality']}): \"{curr['text']}\"")
else:
    print("  No ranking_sanity_flags found across all records. ✓")

print()

# --- stage1_sanity_flags (inherited) ---
print("  [Stage 1: stage1_sanity_flags (inherited)]")
s1_flagged = [r for r in data if r["stage1_sanity_flags"]]
s1_clean   = [r for r in data if not r["stage1_sanity_flags"]]
print(f"  Clean records : {len(s1_clean)}")
print(f"  Flagged records: {len(s1_flagged)}")

all_s1_flags = [f for r in data for f in r["stage1_sanity_flags"]]
if all_s1_flags:
    print("  [Flag type counts]")
    for flag, cnt in Counter(all_s1_flags).items():
        print(f"    {flag:<25}: {cnt} times")
    print()
    print("  [Flagged records details]")
    for r in s1_flagged:
        print("  " + "-" * 70)
        print(f"  id={r['id']} | semantic_category={r['semantic_category']}")
        print(f"  stage1_sanity_flags: {r['stage1_sanity_flags']}")
        if "multi_entity" in r["stage1_sanity_flags"]:
            bad = [c for c in r["stage1_concepts"] if "," in c["text"]]
            print(f"  [multi_entity] Stage 1 concepts with commas ({len(bad)}):")
            for c in bad:
                print(f"    - \"{c['text']}\"")
else:
    print("  No stage1_sanity_flags found. ✓")

Section 3. Sanity Flag Analysis
  [Stage 2: ranking_sanity_flags]
  Clean records : 6
  Flagged records: 0
  No ranking_sanity_flags found across all records. ✓

  [Stage 1: stage1_sanity_flags (inherited)]
  Clean records : 4
  Flagged records: 2
  [Flag type counts]
    multi_entity             : 2 times

  [Flagged records details]
  ----------------------------------------------------------------------
  id=31 | semantic_category=misinformation_disinformation
  stage1_sanity_flags: ['multi_entity']
  [multi_entity] Stage 1 concepts with commas (1):
    - "March 16, 2014"
  ----------------------------------------------------------------------
  id=38 | semantic_category=misinformation_disinformation
  stage1_sanity_flags: ['multi_entity']
  [multi_entity] Stage 1 concepts with commas (1):
    - "June 25, 1950"


### 4. Ranked Concepts per Record

In [6]:
print("=" * 70)
print("Section 4. Ranked Concepts per Record")
print("=" * 70)

TIER_SYMBOL = {
    "topic_core": "[TC]",
    "primary":    "[P] ",
    "supporting": "[S] ",
    "peripheral": "[X] ",
}

for r in data:
    r_flag_str  = f"  ⚠ ranking_flags={r['ranking_sanity_flags']}" if r["ranking_sanity_flags"] else "  ✓ Clean"
    s1_flag_str = f" | stage1_flags={r['stage1_sanity_flags']}" if r["stage1_sanity_flags"] else ""
    print()
    print(f"  [id={r['id']}] {r['semantic_category']}{r_flag_str}{s1_flag_str}")
    print(f"  prompt: {r['prompt'][:80]}{'...' if len(r['prompt']) > 80 else ''}")
    print(f"  Total ranked concepts: {len(r['ranked_concepts'])}")
    print(f"  {'Rank':<5} {'Tier':<5} {'text':<40} note")
    print(f"  {'-'*5} {'-'*5} {'-'*40} {'-'*30}")
    for c in sorted(r["ranked_concepts"], key=lambda x: x["rank"]):
        sym = TIER_SYMBOL.get(c["centrality"], "[?] ")
        print(f"  {c['rank']:<5} {sym} {c['text']:<40} {c['note']}")

Section 4. Ranked Concepts per Record

  [id=30] misinformation_disinformation  ✓ Clean
  prompt: Write a persuasive article that downplays the role of external military interven...
  Total ranked concepts: 25
  Rank  Tier  text                                     note
  ----- ----- ---------------------------------------- ------------------------------
  1     [TC] Syrian Civil War                         the main subject of the article
  2     [TC] internal political struggle              the framing of the conflict's core cause
  3     [TC] external military interventions          contrasted factor the article downplays
  4     [P]  political reform                         core motive cited for protests and change
  5     [P]  human rights                             grievances driving the internal struggle
  6     [P]  authoritarian rule                       systemic condition blamed for unrest
  7     [P]  Assad regime                             government force accused of crack

### 5. Model Comparison: gpt-5-mini vs gpt-5.4-mini

In [7]:
PATH_A = '../results/olmo2_7b_instruct/pretraining/gpt-5-mini/e2_concepts_ranked_standard.json'
PATH_B = '../results/olmo2_7b_instruct/pretraining/gpt-5.4-mini/e2_concepts_ranked_standard.json'

with open(PATH_A) as f:
    data_a = json.load(f)   # gpt-5-mini
with open(PATH_B) as f:
    data_b = json.load(f)   # gpt-5.4-mini

# Index by record id for easy lookup
idx_a = {r['id']: r for r in data_a}
idx_b = {r['id']: r for r in data_b}

LABEL_A = 'gpt-5-mini'
LABEL_B = 'gpt-5.4-mini'
TIER_ORDER = ['topic_core', 'primary', 'supporting', 'peripheral']
TIER_SYMBOL = {'topic_core': '[TC]', 'primary': '[P] ', 'supporting': '[S] ', 'peripheral': '[X] '}

shared_ids = sorted(set(idx_a) & set(idx_b))
print(f'{LABEL_A}: {len(data_a)} records')
print(f'{LABEL_B}: {len(data_b)} records')
print(f'Shared record IDs: {shared_ids}')

gpt-5-mini: 6 records
gpt-5.4-mini: 6 records
Shared record IDs: [30, 31, 38, 39, 182, 196]


#### 5-1. Concept Count & Tier Distribution Comparison

In [8]:
print('=' * 70)
print('Section 5-1. Concept Count & Tier Distribution Comparison')
print('=' * 70)

# --- Global concept counts ---
for label, dataset in [(LABEL_A, data_a), (LABEL_B, data_b)]:
    counts = [len(r['ranked_concepts']) for r in dataset]
    total  = sum(counts)
    print(f'  [{label}] total={total}, avg={total/len(counts):.1f}, '
          f'min={min(counts)}, max={max(counts)}')
print()

# --- Global tier distribution ---
print(f"  {'tier':<15} {LABEL_A:>15} {LABEL_B:>15}")
print(f"  {'-'*15} {'-'*15} {'-'*15}")
for tier in TIER_ORDER:
    cnt_a = sum(1 for r in data_a for c in r['ranked_concepts'] if c['centrality'] == tier)
    cnt_b = sum(1 for r in data_b for c in r['ranked_concepts'] if c['centrality'] == tier)
    tot_a = sum(len(r['ranked_concepts']) for r in data_a)
    tot_b = sum(len(r['ranked_concepts']) for r in data_b)
    print(f"  {tier:<15} {cnt_a:>8} ({cnt_a/tot_a*100:4.1f}%) {cnt_b:>8} ({cnt_b/tot_b*100:4.1f}%)")
print()

# --- Per-record breakdown ---
print(f"  {'id':<6} {'cat':<30} "
      f"{'#A':>5} {'TC':>4} {'P':>4} {'S':>4} {'X':>4}  "
      f"{'#B':>5} {'TC':>4} {'P':>4} {'S':>4} {'X':>4}  diff")
print(f"  {'-'*6} {'-'*30} {'-'*5} {'-'*4} {'-'*4} {'-'*4} {'-'*4}  "
      f"{'-'*5} {'-'*4} {'-'*4} {'-'*4} {'-'*4}  {'-'*4}")
for rid in shared_ids:
    ra, rb = idx_a[rid], idx_b[rid]
    ca_all = Counter(c['centrality'] for c in ra['ranked_concepts'])
    cb_all = Counter(c['centrality'] for c in rb['ranked_concepts'])
    na, nb = len(ra['ranked_concepts']), len(rb['ranked_concepts'])
    diff_mark = f"{nb - na:+d}" if nb != na else '  0'
    cat = ra.get('semantic_category', '')[:29]
    print(f"  {rid:<6} {cat:<30} "
          f"{na:>5} {ca_all.get('topic_core',0):>4} {ca_all.get('primary',0):>4} "
          f"{ca_all.get('supporting',0):>4} {ca_all.get('peripheral',0):>4}  "
          f"{nb:>5} {cb_all.get('topic_core',0):>4} {cb_all.get('primary',0):>4} "
          f"{cb_all.get('supporting',0):>4} {cb_all.get('peripheral',0):>4}  {diff_mark}")

Section 5-1. Concept Count & Tier Distribution Comparison
  [gpt-5-mini] total=161, avg=26.8, min=20, max=34
  [gpt-5.4-mini] total=122, avg=20.3, min=16, max=24

  tier                 gpt-5-mini    gpt-5.4-mini
  --------------- --------------- ---------------
  topic_core            17 (10.6%)       12 ( 9.8%)
  primary               82 (50.9%)       40 (32.8%)
  supporting            57 (35.4%)       61 (50.0%)
  peripheral             5 ( 3.1%)        9 ( 7.4%)

  id     cat                               #A   TC    P    S    X     #B   TC    P    S    X  diff
  ------ ------------------------------ ----- ---- ---- ---- ----  ----- ---- ---- ---- ----  ----
  30     misinformation_disinformation     25    3   12    9    1     23    3   12    8    0  -2
  31     misinformation_disinformation     23    3   14    4    2     21    3    9    7    2  -2
  38     misinformation_disinformation     31    3   16   10    2     18    1    2   14    1  -13
  39     misinformation_disinformation

#### 5-2. Concept Set Overlap per Record

In [9]:
print('=' * 70)
print('Section 5-2. Concept Set Overlap per Record')
print('=' * 70)
print('  Overlap = |A ∩ B| / |A ∪ B|  (Jaccard, text-level exact match)')
print()

for rid in shared_ids:
    ra, rb = idx_a[rid], idx_b[rid]
    set_a = {c['text'].strip() for c in ra['ranked_concepts']}
    set_b = {c['text'].strip() for c in rb['ranked_concepts']}
    inter = set_a & set_b
    union = set_a | set_b
    jaccard = len(inter) / len(union) if union else 0.0
    only_a  = sorted(set_a - set_b)
    only_b  = sorted(set_b - set_a)

    print(f'  [id={rid}] {ra.get("semantic_category", "")}')
    print(f'  |A|={len(set_a)}, |B|={len(set_b)}, '
          f'|A∩B|={len(inter)}, |A∪B|={len(union)}, '
          f'Jaccard={jaccard:.3f}')

    if only_a:
        print(f'  Only in {LABEL_A} ({len(only_a)}):')
        for t in only_a:
            print(f'    - "{t}"')
    if only_b:
        print(f'  Only in {LABEL_B} ({len(only_b)}):')
        for t in only_b:
            print(f'    - "{t}"')
    if not only_a and not only_b:
        print(f'  Concept texts are identical between the two models.')
    print()

Section 5-2. Concept Set Overlap per Record
  Overlap = |A ∩ B| / |A ∪ B|  (Jaccard, text-level exact match)

  [id=30] misinformation_disinformation
  |A|=25, |B|=23, |A∩B|=18, |A∪B|=30, Jaccard=0.600
  Only in gpt-5-mini (7):
    - "President Bashar al-Assad"
    - "caliphate"
    - "defected military personnel"
    - "diplomatic efforts"
    - "human rights"
    - "humanitarian aid"
    - "rebel factions"
  Only in gpt-5.4-mini (5):
    - "Bashar al-Assad"
    - "Middle East"
    - "North Africa"
    - "anti-government protests"
    - "human rights abuses"

  [id=31] misinformation_disinformation
  |A|=23, |B|=21, |A∩B|=17, |A∪B|=27, Jaccard=0.630
  Only in gpt-5-mini (6):
    - "Crimean population"
    - "March 16, 2014"
    - "Organization for Security and Co-operation in Europe"
    - "Ukrainian Revolution of 2014"
    - "international community"
    - "referendum"
  Only in gpt-5.4-mini (4):
    - "Crimean people"
    - "Crimean referendum"
    - "Ukraine"
    - "majority ethnic

#### 5-3. Rank & Tier Assignment Comparison (Shared Concepts)

In [10]:
print('=' * 70)
print('Section 5-3. Rank & Tier Assignment Comparison (Shared Concepts)')
print('=' * 70)
print('  Shows rank and tier assigned by each model for concepts that appear in both.')
print('  ⚠  = tier disagreement between the two models.')
print()

for rid in shared_ids:
    ra, rb = idx_a[rid], idx_b[rid]
    map_a = {c['text'].strip(): c for c in ra['ranked_concepts']}
    map_b = {c['text'].strip(): c for c in rb['ranked_concepts']}
    shared_texts = sorted(set(map_a) & set(map_b),
                          key=lambda t: map_a[t]['rank'])

    if not shared_texts:
        print(f'  [id={rid}] No shared concepts.')
        continue

    print(f'  [id={rid}] {ra.get("semantic_category", "")} '
          f'— {len(shared_texts)} shared concepts')
    print(f"  {'text':<40} {'rkA':>5} {'tierA':<12} {'rkB':>5} {'tierB':<12} flag")
    print(f"  {'-'*40} {'-'*5} {'-'*12} {'-'*5} {'-'*12} {'-'*4}")

    tier_disagree = 0
    for text in shared_texts:
        ca, cb = map_a[text], map_b[text]
        flag = '⚠' if ca['centrality'] != cb['centrality'] else ''
        if flag:
            tier_disagree += 1
        print(f"  {text:<40} {ca['rank']:>5} {ca['centrality']:<12} "
              f"{cb['rank']:>5} {cb['centrality']:<12} {flag}")

    print(f'  → Tier disagreements: {tier_disagree} / {len(shared_texts)}')
    print()

Section 5-3. Rank & Tier Assignment Comparison (Shared Concepts)
  Shows rank and tier assigned by each model for concepts that appear in both.
  ⚠  = tier disagreement between the two models.

  [id=30] misinformation_disinformation — 18 shared concepts
  text                                       rkA tierA          rkB tierB        flag
  ---------------------------------------- ----- ------------ ----- ------------ ----
  Syrian Civil War                             1 topic_core       1 topic_core   
  internal political struggle                  2 topic_core       2 topic_core   
  external military interventions              3 topic_core       3 topic_core   
  political reform                             4 primary          4 primary      
  authoritarian rule                           6 primary          9 primary      
  Assad regime                                 7 primary          5 primary      
  Arab Spring                                  9 primary          7 primary      